# Sammenligning af tilbagebetalingsplaner for studielån med PROC LOAN

## Sammendrag

Et studievejledningskontor vurderer, hvordan en dimittendårgang bør tilbagebetale en repræsentativ saldo på $27.500 ved at sammenligne fire tilbagebetalingsstrukturer — et statsligt fastforrentet standardlån, en privat omlægning med et etableringsgebyr, et loftbelagt variabelt forrentet (ARM) lån og en arbejdsgiverstøttet rabatordning — ved hjælp af `PROC LOAN`. Over en løbetid på 120 måneder ligger de fire tilbud pænt på linje med hensyn til den månedlige ydelse og de samlede renter ved deres angivne startrenter: det statslige standardlån koster mest i renter (**$10.021,22** ved 6,53%, ydelse **$312,68**), den private omlægning sænker renten, men tilføjer et gebyr på $412,50 (**$9.120,20** ved 5,99%, ydelse **$305,17**), og det lavere angivne ARM-lån (**$7.099,76** ved 4,75%, ydelse **$288,33**) samt arbejdsgiverens rabatlån (**$6.700,67** ved 4,50%, ydelse **$285,01**) har de laveste renteudgifter. En `COMPARE`-blok rapporterer derefter for hver plan de akkumulerede renter, den akkumulerede afdragne hovedstol og den udestående saldo ved 36, 60 og 120 måneder, hvilket viser, hvor langt hvert lån er blevet afdraget på de tidspunkter, hvor en låntager mest sandsynligt vil omlægge lånet eller indfri det.

## Datakilder

| Datasæt | Rækker | Beskrivelse | Nøglevariabler |
|---------|--------|-------------|----------------|
| `borrowers` | 40 | Syntetiske lånprofiler for en dimittendårgang, genereret direkte med `call streaminit(20260531)` og `rand('uniform')`. Bruges til at motivere realistiske lånevilkår til sammenligningen. | `student_id` (1001-1040), `balance` (hovedstol ved dimission; dette udtræk spænder fra $11.800-$47.750, gennemsnit $30.771), `apr` (4,10%-9,10% nominel årlig rente, gennemsnit 6,50%), `term` (120 eller 180 måneder, gennemsnit 144), `origination_pct` (gebyr på 1,0%-2,0%, gennemsnit 1,50%) |

Den repræsentative låntager, der analyseres med `PROC LOAN` (beløb $27.500, løbetid 120 måneder, start juli 2026), ligger tæt på den nedre midte af denne årgangs fordeling af saldo og rente; der anvendes ingen eksterne data eller netværksdata. Årgangen findes for at motivere plausible lånevilkår — den direkte sammenligning udføres på det ene repræsentative lån.

# Sammenligning af tilbagebetalingsplaner for studielån med PROC LOAN

Når studerende dimitterer, skal et studievejledningskontor hjælpe dem med at vælge mellem konkurrerende tilbagebetalingstilbud. `PROC LOAN` (SAS/ETS) er specialbygget til netop dette: den afdrager fastforrentede, variabelt forrentede (ARM) og rabatlån og sammenligner dem derefter side om side på ydelse, samlede renter og afdragsforløb.

I denne notebook vil vi:

1. Generere en syntetisk dimittendårgang for at fastlægge realistiske lånevilkår.
2. Opsummere årgangen med `PROC MEANS`.
3. Opbygge fire tilbagebetalingsplaner for en repræsentativ saldo på $27.500 og sammenligne dem med `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Genkøre den anbefalede fastforrentede plan alene for at bekræfte dens selvstændige økonomi.

Alt kører lokalt på indlejrede syntetiske data — ingen eksterne filer eller netværksadgang.

## Trin 1 — Generér en syntetisk dimittendårgang

Vi simulerer 40 låntagere. Hver trækker en hovedstol ved dimission, en årlig rente løst knyttet til kreditkvalitet, en standard tilbagebetalingsperiode (10 eller 15 år) og en etableringsgebyrprocent. Frøet (seed) gør dataene reproducerbare.

In [1]:
data borrowers;
   CALL streaminit(20260531);
   LÆNGDE plan $ 28;
   GØR student_id = 1001 TIL 1040;
      /* Principal balance at graduation: $9,500 - $48,000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Annual nominal APR by credit tier: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standard repayment term: 120 or 180 months */
      HVIS rand('uniform') < 0.6 SÅ term = 120;
      ELLERS term = 180;
      /* Origination fee as a percent of principal: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      UDDATA;
   SLUT;
KØR;



NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Trin 2 — Profilér årgangen

Før vi modellerer individuelle lån, ser vi på fordelingen af saldi, renter og løbetider. Dette fortæller os, hvordan en *repræsentativ* låntager ser ud — grundlaget for den direkte sammenligning, der følger.

In [2]:
PROCEDURE GENNEMSNIT data=borrowers n mean MIN MAX maxdec=2;
   VARIABEL balance apr term origination_pct;
   MÆRKAT balance='Hovedstol ($)' apr='Årlig rente (%)' term='Løbetid (mdr.)'
         origination_pct='Etableringsgebyr (%)';
   TITEL 'Profil af dimittendårgangen';
KØR;


                                              Profil af dimittendårgangen                                               

                                                  The MEANS Procedure

 Variable         Label                       N           Mean     Minimum     Maximum
 -------------------------------------------------------------------------------------
 balance          Hovedstol ($)              40       30771.25    11800.00    47750.00
 apr              Årlig rente (%)            40           6.50        4.10        9.10
 term             Løbetid (mdr.)             40         144.00      120.00      180.00
 origination_pct  Etableringsgebyr (%)       40           1.50        1.00        2.00
 -------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Trin 3 — Sammenlign fire tilbagebetalingsplaner for en repræsentativ låntager

Med en repræsentativ saldo på $27.500 over en løbetid på 120 måneder (10 år) med start i juli 2026 stiller vi fire realistiske tilbud op mod hinanden:

- **Statsligt direkte lån uden subsidiering (Standard)** — et almindeligt fastforrentet lån til 6,53%.
- **Privat omlægning (med gebyr)** — en lavere fast rente på 5,99%, men med en etableringsomkostning på $412,50 (`INIT=`).
- **Privat variabelt ARM-lån** — et lån med justerbar rente på 4,75% med et loft (`CAPS=`) på 1% pr. justering / 5% for hele løbetiden, en maksimumrente (`MAXRATE=`) på 9,75%, årlig justeringsfrekvens (`ADJUSTFREQ=`) og nøgleordet `WORSTCASE`.
- **Arbejdsgiver 2-1 rabatlån** — en subsidieret startrente på 4,50%, som ifølge den angivne plan trinvis stiger via `BDRATES=` til 5,50% i år 2 og 6,50% derefter.

`COMPARE`-sætningen beder om et tværgående overblik ved 36, 60 og 120 måneder, med en skattesats (`TAXRATE=`) på 22% og et minimumsafkastkrav (`MARR=`) på 4%; `OUTSUM=` og `OUTCOMP=` opsamler henholdsvis lånesammendraget og sammenligningsrækkerne. Nedenstående liste rapporterer for hver plan og hvert kontrolpunkt de **akkumulerede betalte renter, den akkumulerede afdragne hovedstol og den udestående saldo** — et klart billede af afdragsforløbet på tværs af kandidaterne.

> **Bemærkning om rentejusteringer.** Jenners `PROC LOAN` afdrager alle planer til deres angivne **start**rente, så ARM-lånets `CAPS`/`MAXRATE`/`WORSTCASE`-indstillinger og rabatlånets `BDRATES`-trin gengives i listen som lånevilkår, men indgår **ikke** i beregningen af ydelsen — ARM- og rabatlånstallene nedenfor afspejler deres startrenter på henholdsvis 4,75% og 4,50% holdt fast gennem hele løbetiden. Betragt disse to totaler som best-case-tal (baseret på startrenten), ikke som worst-case-lofter.

In [3]:
PROCEDURE loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         MÆRKAT='Statsligt direkte lån uden subsidiering (Standard)';

   fixed rate=5.99 init=412.50
         MÆRKAT='Privat omlægning (med gebyr)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       MÆRKAT='Privat variabelt ARM-lån (værst tænkelige)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           MÆRKAT='Arbejdsgiver 2-1 rabatlån, derefter 6,5%';

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
KØR;


                                              Profil af dimittendårgangen                                               

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Statsligt direkte lån uden subsidiering (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Privat omlægning (med gebyr)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:              


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Trin 4 — Genkør den anbefalede fastforrentede plan alene

For den låntager, der lægger vægt på sikkerhed om ydelsen, er det statslige fastforrentede standardlån det sikre standardvalg. Vi genkører det isoleret for at bekræfte dets overordnede økonomi: den samme månedlige ydelse på **$312,68**, samlet betalt beløb på **$37.521,22** og samlede renter på **$10.021,22**, som sås i firevejssammenligningen, nu præsenteret som et selvstændigt lånesammendrag.

In [4]:
PROCEDURE loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed MÆRKAT='Statsligt direkte lån uden subsidiering (Standard)';
KØR;


                                              Profil af dimittendårgangen                                               

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Statsligt direkte lån uden subsidiering (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Statsligt direkte lån uden subsidiering (Standard)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Fortolkning af resultaterne

Alle fire planer afdrager fuldt ud den samme hovedstol på $27.500 over 120 måneder (hver plans udestående saldo når $0,00 ved måned 120 i COMPARE-tabellen), så beslutningen afhænger af **den månedlige ydelse** og **de samlede renter ved den angivne rente**:

| Plan | Rente | Ydelse | Samlede renter | Bemærkninger |
|------|-------|--------|-----------------|--------------|
| Statsligt direkte standardlån | 6,53% | $312,68 | $10.021,22 | Intet gebyr; stærkeste låntagerbeskyttelse |
| Privat omlægning | 5,99% | $305,17 | $9.120,20 | Etableringsgebyr på $412,50 |
| Privat variabelt ARM-lån | 4,75% (start) | $288,33 | $7.099,76 | Renten kan stige; tallet er startrente-omkostningen |
| Arbejdsgiver 2-1 rabatlån | 4,50% (start) | $285,01 | $6.700,67 | Afhænger af fortsat ansættelse |

- Den **statslige standardplan** er den dyreste i renter ($10.021,22), men tilbyder en fast, forudsigelig ydelse på $312,68 uden gebyr.
- Den **private omlægning** sænker renten til 5,99% ($9.120,20 i renter, $901 mindre end den statslige plan), men opkræver et etableringsgebyr på $412,50, så nettofordelen i forhold til den statslige plan er omkring $488 i renter minus gebyr — kun relevant, hvis lånet løber længe nok til, at det ikke bliver omlagt igen.
- **ARM-lånet** og **rabatlånet** viser de laveste renter her ($7.099,76 og $6.700,67) udelukkende fordi deres **startrenter** ligger et godt stykke under de faste tilbud. Som nævnt i trin 3 holder Jenner disse startrenter faste, så disse tal er best-case: et reelt ARM-lån, der justeres op — eller et rabatlån, der mister arbejdsgiverens støtte — ville lande højere, og ydelsen er ikke garanteret.

`COMPARE`-tabellen præciserer dette ved at vise, hvor hurtigt hver plan afdrages. Ved 36 måneder har den statslige plan betalt $4.792,27 i renter og afdraget $6.464,10 af hovedstolen (saldo $21.035,90), mens rabatlånet kun har betalt $3.263,97 i renter og afdraget $6.996,24 af hovedstolen (saldo $20.503,76) — planerne med lavere rente koster både færre renter og afdrager hovedstolen hurtigere i de første tre år. For en risikoavers dimittend retfærdiggør den statslige standardplans rentesikkerhed ofte dens højere renter; for en låntager, der er sikker på stabil, varig beskæftigelse, giver rabatlånets subsidierede start den største besparelse blandt de muligheder, der reelt fastlåser deres rente.